# ATN Training — Cloud GPU Edition
### Works on: Kaggle (free P100) · Google Colab (free T4) · RunPod · Vast.ai

**How to use:**
1. Upload your `atn_train/` and `atn_val/` image folders as a Kaggle Dataset
   *(or mount Google Drive on Colab — instructions in Cell 0)*
2. Set `PLATFORM` in Cell 1 to `'kaggle'` or `'colab'`
3. Run all cells — training takes ~5–10 min/epoch on a free GPU
4. Download `reface_atn.pth` from the output section when done

## Cell 0 — Platform Setup Instructions

### Kaggle (recommended — free, no session limit, P100 GPU)
1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets) → **New Dataset**
2. Upload a `.zip` of your `atn_train/` folder → name it `atn-train-data`
3. Upload a `.zip` of your `atn_val/` folder → name it `atn-val-data`
4. Create a new **Notebook** → add both datasets via **+ Add Data**
5. Set accelerator to **GPU P100** in Settings → right panel
6. Paste this notebook and run

### Google Colab (free T4, 12h session)
1. Open [colab.research.google.com](https://colab.research.google.com)
2. Runtime → Change runtime type → **T4 GPU**
3. Upload images to Google Drive in folders `atn_train/` and `atn_val/`
4. Set `PLATFORM = 'colab'` in Cell 1 — it will auto-mount Drive

### RunPod / Vast.ai (paid, ~$0.20/hr, most control)
1. Rent a pod with PyTorch image (RTX 3090 recommended)
2. `scp` your data folders to the pod
3. Set `PLATFORM = 'runpod'` in Cell 1 and set paths manually

In [ ]:
# Cell 1 - Platform Config & Imports
# ── SET THIS before running ───────────────────────────────────────────────
PLATFORM = 'kaggle'   # options: 'kaggle' | 'colab' | 'runpod'
# ─────────────────────────────────────────────────────────────────────────

import os, gc, math, json, time, hashlib, subprocess, sys
from pathlib import Path

# Install any missing packages
def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

pip('facenet-pytorch', 'scikit-image', 'tqdm')

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
from skimage.metrics import structural_similarity as ssim
from torch.cuda.amp import autocast, GradScaler

# ── Paths per platform ────────────────────────────────────────────────────
if PLATFORM == 'kaggle':
    # Kaggle datasets are mounted at /kaggle/input/<dataset-slug>/
    # Adjust the slugs below to match what you named your datasets
    TRAIN_DIR = Path('/kaggle/input/atn-train-data/atn_train')
    VAL_DIR   = Path('/kaggle/input/atn-val-data/atn_val')
    OUT_DIR   = Path('/kaggle/working')

elif PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    # Change these to wherever you put the folders in Drive
    TRAIN_DIR = Path('/content/drive/MyDrive/atn_train')
    VAL_DIR   = Path('/content/drive/MyDrive/atn_val')
    OUT_DIR   = Path('/content')

elif PLATFORM == 'runpod':
    TRAIN_DIR = Path('/workspace/data/atn_train')
    VAL_DIR   = Path('/workspace/data/atn_val')
    OUT_DIR   = Path('/workspace/models')

else:
    raise ValueError(f'Unknown PLATFORM: {PLATFORM}')

OUTPUT_PATH    = OUT_DIR / 'reface_atn.pth'
CHECKPOINT_DIR = OUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────────────────
EPOCHS         = 20
LR             = 1e-4
EPSILON        = 0.05
LAMBDA_X       = 1.0
LAMBDA_Y       = 10.0
IMG_SIZE       = 224
BATCH_SIZE     = 64     # GPU can handle large batches — much faster
NUM_WORKERS    = 4

# ── Device ────────────────────────────────────────────────────────────────
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'

print('='*55)
print(f'  Platform  : {PLATFORM}')
print(f'  Device    : {DEVICE}')
if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'  GPU       : {props.name}')
    print(f'  VRAM      : {props.total_memory/1e9:.1f} GB')
print(f'  AMP       : {USE_AMP}')
print(f'  Batch     : {BATCH_SIZE}')
print(f'  Img size  : {IMG_SIZE}x{IMG_SIZE}')
print(f'  Epochs    : {EPOCHS}')
print(f'  Train dir : {TRAIN_DIR}')
print(f'  Val dir   : {VAL_DIR}')
print(f'  Output    : {OUTPUT_PATH}')
print('='*55)
assert TRAIN_DIR.exists(), f'TRAIN_DIR not found: {TRAIN_DIR}'
assert VAL_DIR.exists(),   f'VAL_DIR not found: {VAL_DIR}'
print('Paths verified ✓')


In [ ]:
# Cell 2 - Dataset Preview
def list_images(d):
    exts = {'.jpg','.jpeg','.png','.bmp','.webp'}
    return [p for p in Path(d).rglob('*') if p.suffix.lower() in exts]

train_images = list_images(TRAIN_DIR)
val_images   = list_images(VAL_DIR)
print(f'Train: {len(train_images)} images')
print(f'Val  : {len(val_images)} images')

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for ax, p in zip(axes.flatten(), train_images[:9]):
    img = Image.open(p).convert('RGB')
    ax.imshow(img); ax.set_title(f'{img.size[0]}x{img.size[1]}'); ax.axis('off')
for ax in axes.flatten()[9:]: ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# Cell 3 - Dataset & DataLoader
class FaceDataset(Dataset):
    def __init__(self, paths, tf): self.paths, self.tf = paths, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.tf(img)

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

train_loader = DataLoader(FaceDataset(train_images, train_tf),
    batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(FaceDataset(val_images,   val_tf),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches/epoch : {len(train_loader)}')
b = next(iter(train_loader))
print(f'First batch shape   : {tuple(b.shape)}')


In [ ]:
# Cell 4 - ATN Architecture (Full-size — GPU can handle it)
class ResidualBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch,ch,3,padding=1,bias=False), nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
            nn.Conv2d(ch,ch,3,padding=1,bias=False), nn.BatchNorm2d(ch)
        )
    def forward(self, x): return F.relu(self.net(x) + x, inplace=True)

class DownBlock(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.c = nn.Sequential(
            nn.Conv2d(ic,oc,3,stride=2,padding=1,bias=False),
            nn.BatchNorm2d(oc), nn.ReLU(inplace=True), ResidualBlock(oc))
    def forward(self, x): return self.c(x)

class UpBlock(nn.Module):
    def __init__(self, ic, sc, oc):
        super().__init__()
        self.up   = nn.ConvTranspose2d(ic, oc, 2, stride=2)
        self.fuse = nn.Sequential(
            nn.Conv2d(oc+sc,oc,3,padding=1,bias=False),
            nn.BatchNorm2d(oc), nn.ReLU(inplace=True), ResidualBlock(oc))
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.fuse(torch.cat([x, skip], dim=1))

class ResidualUNetATN(nn.Module):
    def __init__(self, eps=0.05):
        super().__init__()
        self.eps        = eps
        self.stem       = nn.Sequential(nn.Conv2d(3,64,3,padding=1,bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.down1      = DownBlock(64, 128)
        self.down2      = DownBlock(128, 256)
        self.down3      = DownBlock(256, 512)
        self.down4      = DownBlock(512, 512)
        self.bottleneck = nn.Sequential(ResidualBlock(512), ResidualBlock(512))
        self.up1        = UpBlock(512, 512, 512)
        self.up2        = UpBlock(512, 256, 256)
        self.up3        = UpBlock(256, 128, 128)
        self.up4        = UpBlock(128,  64,  64)
        self.out        = nn.Conv2d(64, 3, 3, padding=1)
    def forward(self, x):
        s0=self.stem(x); s1=self.down1(s0); s2=self.down2(s1)
        s3=self.down3(s2); s4=self.down4(s3); b=self.bottleneck(s4)
        d1=self.up1(b,s3); d2=self.up2(d1,s2); d3=self.up3(d2,s1); d4=self.up4(d3,s0)
        return torch.clamp(torch.tanh(self.out(d4)), -self.eps, self.eps)

atn = ResidualUNetATN(eps=EPSILON).to(DEVICE)
total_p = sum(p.numel() for p in atn.parameters())
print(f'ATN parameters : {total_p:,}')
print(f'ATN device     : {next(atn.parameters()).device}')


In [ ]:
# Cell 5 - Surrogate (on GPU — CUDA has plenty of VRAM, both models fit fine)
surrogate = InceptionResnetV1(pretrained='vggface2').eval().to(DEVICE)
for p in surrogate.parameters(): p.requires_grad = False
print(f'Surrogate : {surrogate.__class__.__name__} on {next(surrogate.parameters()).device}')
if DEVICE.type == 'cuda':
    used = torch.cuda.memory_allocated()/1e9
    total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'VRAM used after loading both models: {used:.2f} / {total:.1f} GB')


In [ ]:
# Cell 6 - Loss Functions
def perturbation_loss(pert):  return torch.mean(pert.pow(2))
def adversarial_loss(ce, se):
    return torch.mean(torch.sum(F.normalize(ce,2,1) * F.normalize(se,2,1), dim=1))
def total_loss(lx, ly):       return LAMBDA_X*lx + LAMBDA_Y*ly
print(f'Losses ready.  λx={LAMBDA_X}  λy={LAMBDA_Y}')


In [ ]:
# Cell 6b - Pre-compute Clean Embeddings (GPU, one-time)
print('Caching clean embeddings...')
clean_cache = {}
surrogate.eval()
with torch.no_grad():
    for idx, imgs in enumerate(tqdm(train_loader, desc='Caching')):
        imgs = imgs.to(DEVICE, non_blocking=True)
        clean_cache[idx] = F.normalize(surrogate(imgs), dim=1).cpu()  # store on CPU

total = sum(v.shape[0] for v in clean_cache.values())
print(f'Cached {total} embeddings. GPU cache cleared.')
torch.cuda.empty_cache()
gc.collect()


In [ ]:
# Cell 7 - Training Loop (Full GPU + AMP)
optimizer = torch.optim.Adam(atn.parameters(), lr=LR)
scaler    = GradScaler(enabled=USE_AMP)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
history   = {'lx':[],'ly':[],'total':[],'embed_dist':[],'epoch_time':[]}

print(f'Starting training for {EPOCHS} epochs on {DEVICE}...')
run_start = time.time()

for epoch in range(1, EPOCHS+1):
    atn.train()
    rlx=rly=rtot=rdist=0.0; nb=0
    t0 = time.time()

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False)
    for bidx, images in enumerate(pbar):
        images   = images.to(DEVICE, non_blocking=True)
        clean_emb = clean_cache[bidx].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=USE_AMP):
            perturb      = atn(images)
            shielded     = torch.clamp(images + perturb, -1.0, 1.0)
            with torch.no_grad():
                shield_emb = surrogate(shielded)
            lx   = perturbation_loss(perturb)
            ly   = adversarial_loss(clean_emb, shield_emb)
            loss = total_loss(lx, ly)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            dist = (1 - F.cosine_similarity(
                F.normalize(clean_emb,2,1),
                F.normalize(shield_emb,2,1))).mean().item()

        rlx+=lx.item(); rly+=ly.item(); rtot+=loss.item(); rdist+=dist; nb+=1
        pbar.set_postfix(Lx=f'{lx.item():.4f}', Ly=f'{ly.item():.4f}',
                         Loss=f'{loss.item():.4f}', D=f'{dist:.4f}')

    scheduler.step()
    et = time.time()-t0

    elx=rlx/nb; ely=rly/nb; etot=rtot/nb; edist=rdist/nb
    history['lx'].append(elx); history['ly'].append(ely)
    history['total'].append(etot); history['embed_dist'].append(edist)
    history['epoch_time'].append(et)

    lr_now = scheduler.get_last_lr()[0]
    print(f'Epoch {epoch:02d} | Lx={elx:.5f} Ly={ely:.5f} '
          f'Total={etot:.5f} EmbDist={edist:.4f} '
          f'LR={lr_now:.2e} Time={et:.0f}s')

    if epoch % 5 == 0:
        ckpt = CHECKPOINT_DIR / f'atn_epoch_{epoch:02d}.pth'
        torch.save({'epoch':epoch,'model_state_dict':atn.state_dict(),
                    'optimizer_state_dict':optimizer.state_dict(),
                    'history':history}, ckpt)
        print(f'  Checkpoint saved → {ckpt.name}')

    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

total_time = time.time()-run_start
print(f'\nTraining complete in {total_time/60:.1f} min')


In [ ]:
# Cell 8 - Training Curves
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Perturbation Loss (Lx)', 'Adversarial Loss (Ly)',
          'Total Loss', 'Embedding Distance']
keys   = ['lx', 'ly', 'total', 'embed_dist']
colors = ['steelblue', 'tomato', 'mediumpurple', 'seagreen']
ep     = range(1, len(history['lx'])+1)
for ax, title, key, color in zip(axes, titles, keys, colors):
    ax.plot(ep, history[key], marker='o', color=color, linewidth=2)
    ax.set_title(title, fontsize=11); ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Best embedding distance: {max(history["embed_dist"]):.4f} '
      f'(epoch {history["embed_dist"].index(max(history["embed_dist"]))+1})')


In [ ]:
# Cell 9 - Visual Validation
atn.eval()
samples = val_images[:5]
fig, axes = plt.subplots(len(samples), 3, figsize=(12, 3*len(samples)))
if len(samples)==1: axes = axes[None]

for i, p in enumerate(samples):
    pil = Image.open(p).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    x   = val_tf(pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pert = atn(x)
        y    = torch.clamp(x + pert, -1.0, 1.0)

    def to_img(t): return ((t[0].cpu().permute(1,2,0).numpy()*0.5)+0.5).clip(0,1)
    xi = to_img(x); yi = to_img(y)
    pi = pert[0].cpu().permute(1,2,0).numpy()*10
    pi = (pi-pi.min())/(pi.max()-pi.min()+1e-8)

    sc = ssim((xi*255).astype('uint8'), (yi*255).astype('uint8'), channel_axis=2)
    print(f'{p.name}: SSIM={sc:.4f}')
    axes[i,0].imshow(xi);  axes[i,0].set_title('Original');       axes[i,0].axis('off')
    axes[i,1].imshow(pi);  axes[i,1].set_title('Perturbation x10'); axes[i,1].axis('off')
    axes[i,2].imshow(yi);  axes[i,2].set_title('Shielded');       axes[i,2].axis('off')

plt.tight_layout(); plt.show()


In [ ]:
# Cell 10 - Embedding Distance Validation
atn.eval()
distances = []
with torch.no_grad():
    for p in tqdm(val_images[:100], desc='Validating'):
        x    = val_tf(Image.open(p).convert('RGB').resize((IMG_SIZE,IMG_SIZE))).unsqueeze(0).to(DEVICE)
        pert = atn(x)
        y    = torch.clamp(x+pert, -1.0, 1.0)
        e1   = F.normalize(surrogate(x), dim=1)
        e2   = F.normalize(surrogate(y), dim=1)
        distances.append((1 - F.cosine_similarity(e1,e2)).item())

arr = np.array(distances)
print(f'Mean embedding distance : {arr.mean():.4f}  (target > 0.30)')
print(f'Std                     : {arr.std():.4f}')
print(f'% pairs distance > 0.5  : {(arr>0.5).mean()*100:.1f}%')

plt.figure(figsize=(8,4))
plt.hist(arr, bins=30, color='steelblue', edgecolor='white')
plt.axvline(arr.mean(), color='red', linestyle='--', label=f'Mean={arr.mean():.3f}')
plt.title('Embedding Distance Distribution'); plt.xlabel('Cosine Distance')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# Cell 11 - Export Model + Download
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(atn.state_dict(), OUTPUT_PATH)

h = hashlib.sha256()
with open(OUTPUT_PATH,'rb') as f:
    for chunk in iter(lambda: f.read(1<<20), b''): h.update(chunk)
sha = h.hexdigest()
size_mb = OUTPUT_PATH.stat().st_size / (1024**2)

print(f'Saved  : {OUTPUT_PATH}')
print(f'Size   : {size_mb:.1f} MB')
print(f'SHA256 : {sha}')

# ── Auto-download depending on platform ──────────────────────────────────
if PLATFORM == 'colab':
    from google.colab import files
    print('Starting browser download...')
    files.download(str(OUTPUT_PATH))

elif PLATFORM == 'kaggle':
    print()
    print('On Kaggle: go to the OUTPUT tab (right panel) → download reface_atn.pth')
    print(f'File is at: {OUTPUT_PATH}')

elif PLATFORM == 'runpod':
    print()
    print('On RunPod: use the Files tab or run this in terminal:')
    print(f'  scp root@<pod-ip>:{OUTPUT_PATH} D:/deepfake_detection/models/reface_atn.pth')

print('\nDone ✓')
